# MFAR V1 — Baseline Reproduction

Notebook pengendali ini mempertahankan Stage 01–07 sebagai notebook terpisah. Dua run lengkap dibuat pada folder terisolasi dan dibandingkan berdasarkan row count, metric, dan SHA-256 CSV utama.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, subprocess, sys
from pathlib import Path

BRANCH = "revision/q1-logic-mfar"
BASELINE_SHA = "125725c10081c4b3511ffc5aaf1b8e0836434bd8"
DRIVE_FOLDER_NAME = "In_Out_MFAR_Modular_Colab_Pipeline"
REPO_URL = "https://github.com/yanto-mashardi/MFAR_Modular_Colab_Pipeline.git"
REPO = Path("/content/MFAR_Modular_Colab_Pipeline_Q1")

if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "switch", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO, check=True)

head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Revision branch:", BRANCH)
print("Current SHA:", head)
print("Frozen scientific baseline SHA:", BASELINE_SHA)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-baseline.txt")], check=True)
sys.path.insert(0, str(REPO))
os.environ["MFAR_CODE_ROOT"] = str(REPO)
from src.mfar_paths import DRIVE_ROOT as SOURCE_ROOT, validate_raw_inputs
validate_raw_inputs("00_Baseline_Reproduction.ipynb")
print("Source Drive root:", SOURCE_ROOT)


In [ ]:
#@title Run tests and execute two isolated Stage 01–07 pipelines
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(REPO / "tests"), "-p", "test_*.py", "-v"], cwd=REPO, check=True)

RUN_PARENT = SOURCE_ROOT / "baseline_reproduction" / BASELINE_SHA
RUN_A = RUN_PARENT / "run_a" / DRIVE_FOLDER_NAME
RUN_B = RUN_PARENT / "run_b" / DRIVE_FOLDER_NAME
NOTEBOOKS = [
    "01_AIS_Input_and_Cleaning.ipynb",
    "02_Time_Grid_and_State_Preparation.ipynb",
    "03_Monitoring_State.ipynb",
    "04_No_Intervention_Forecast.ipynb",
    "05_Fuzzification.ipynb",
    "06_Rule_Evaluation.ipynb",
    "07_Candidate_Action.ipynb",
]

def run_pipeline(run_root: Path):
    if run_root.name != DRIVE_FOLDER_NAME:
        raise ValueError(f"Nama root harus {DRIVE_FOLDER_NAME}: {run_root}")
    if run_root.parent.exists():
        shutil.rmtree(run_root.parent)
    (run_root / "data_raw").mkdir(parents=True)
    executed = run_root / "executed_notebooks"
    executed.mkdir(parents=True)
    for name in ["ais_raw.csv", "vehicle_arrival_rate_30min.csv"]:
        shutil.copy2(SOURCE_ROOT / "data_raw" / name, run_root / "data_raw" / name)
    env = os.environ.copy()
    env["MFAR_CODE_ROOT"] = str(REPO)
    env["MFAR_GDRIVE_ROOT"] = str(run_root)
    env["PYTHONPATH"] = f"{REPO}:{env.get('PYTHONPATH', '')}"
    for stage, notebook in enumerate(NOTEBOOKS, 1):
        print(f"\n{'='*72}\nRUN {run_root.parent.name} — STAGE {stage:02d}: {notebook}\n{'='*72}", flush=True)
        subprocess.run(["jupyter", "nbconvert", "--to", "notebook", "--execute", str(REPO / "notebooks" / notebook), "--output", notebook, "--output-dir", str(executed), "--ExecutePreprocessor.timeout=-1", "--ExecutePreprocessor.kernel_name=python3"], cwd=REPO, env=env, check=True)
    freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze", "--all"], text=True)
    (run_root / "baseline-python-freeze.txt").write_text(freeze, encoding="utf-8")
    evidence = run_root / "baseline_evidence"
    subprocess.run([sys.executable, str(REPO / "tools" / "create_baseline_manifest.py"), "snapshot", "--repo-root", str(REPO), "--drive-root", str(run_root), "--output-dir", str(evidence), "--baseline-sha", BASELINE_SHA, "--branch", BRANCH], cwd=REPO, env=env, check=True)
    return evidence

EVIDENCE_A = run_pipeline(RUN_A)
EVIDENCE_B = run_pipeline(RUN_B)


In [ ]:
#@title Compare complete runs
REPORT = RUN_PARENT / "repeatability_report.json"
subprocess.run([sys.executable, str(REPO / "tools" / "create_baseline_manifest.py"), "compare", "--first", str(EVIDENCE_A / "baseline_manifest.json"), "--second", str(EVIDENCE_B / "baseline_manifest.json"), "--output", str(REPORT), "--tolerance", "1e-12", "--require-canonical-hashes"], cwd=REPO, check=True)
import json
report = json.loads(REPORT.read_text(encoding="utf-8"))
print("Repeatability status:", report["status"])
print("Evidence A:", EVIDENCE_A)
print("Evidence B:", EVIDENCE_B)
print("Comparison:", REPORT)
assert report["status"] == "PASS"
